# SFT Fine-Tuning: Ministral 3 3B on Medical Flashcards

This notebook demonstrates **Supervised Fine-Tuning (SFT)** of `mistralai/Ministral-3-3B-Instruct-2512` on a medical Q&A dataset using Training Hub.

**Goal:** Teach the model to produce concise, factual medical answers in the style of clinical flashcards, rather than verbose general-purpose responses.

**Dataset:** [medalpaca/medical_meadow_medical_flashcards](https://huggingface.co/datasets/medalpaca/medical_meadow_medical_flashcards) — 33,955 medical Q&A pairs covering anatomy, pharmacology, pathology, and clinical medicine.

**Model:** Ministral 3 3B is a compact instruction-tuned model from Mistral AI. Although it originates from a VLM wrapper (`Mistral3ForConditionalGeneration`), Training Hub extracts and trains the CausalLM text backbone (`Ministral3ForCausalLM`) directly.

**Hardware requirements:**
- Minimum: 2× GPUs with 40GB+ VRAM (e.g., A100-40GB, L40S)
- Recommended: 8× A100-80GB for faster training

**Prerequisites:**
```bash
pip install -e . && pip install -e .[cuda] --no-build-isolation
```

## Setup and Imports

In [ ]:
from training_hub import sft

import glob
import json
import os
import time
import logging
import sys
from io import StringIO
from contextlib import redirect_stdout, redirect_stderr

## Logging Configuration

Suppress verbose output from underlying libraries to keep the notebook clean.

In [ ]:
logging.basicConfig(level=logging.WARNING, format='%(name)s - %(levelname)s - %(message)s')
for logger_name in ['transformers', 'datasets', 'torch', 'accelerate']:
    logging.getLogger(logger_name).setLevel(logging.ERROR)

## Utility Functions

In [ ]:
def find_most_recent_checkpoint(output_dir):
    """Find the most recent HF-format checkpoint in the training output directory."""
    checkpoint_dirs = glob.glob(os.path.join(output_dir, "hf_format", "samples_*"))
    if not checkpoint_dirs:
        return None
    return max(checkpoint_dirs, key=os.path.getctime)

## Load and Prepare the Dataset

The [medical_meadow_medical_flashcards](https://huggingface.co/datasets/medalpaca/medical_meadow_medical_flashcards) dataset contains concise medical Q&A pairs. We convert them to the JSONL **messages** format expected by Training Hub.

| Field | Example |
|-------|--------|
| `input` | *What is the mechanism of action of ACE inhibitors?* |
| `output` | *ACE inhibitors block the conversion of angiotensin I to angiotensin II, reducing vasoconstriction and aldosterone secretion.* |

In [ ]:
from datasets import load_dataset

ds = load_dataset("medalpaca/medical_meadow_medical_flashcards", split="train")
print(f"Full dataset: {len(ds):,} samples")

# Use a subset for this demo
SUBSET_SIZE = 3000
ds_subset = ds.shuffle(seed=42).select(range(SUBSET_SIZE))

# Convert to JSONL messages format
DATA_PATH = "./ministral_sft_output/medical_flashcards.jsonl"
os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)

with open(DATA_PATH, "w") as f:
    for row in ds_subset:
        messages = [
            {"role": "user", "content": row["input"]},
            {"role": "assistant", "content": row["output"]},
        ]
        f.write(json.dumps({"messages": messages}) + "\n")

print(f"Wrote {SUBSET_SIZE} samples to {DATA_PATH}")

# Show a few examples
for i, row in enumerate(ds_subset.select(range(3))):
    print(f"\n--- Sample {i} ---")
    print(f"  Q: {row['input'][:120]}")
    print(f"  A: {row['output'][:120]}")

## Configuration

Adjust `NUM_GPUS` and `MAX_TOKENS_PER_GPU` to match your hardware. The defaults below work on 8× A100-80GB.

In [ ]:
import torch

# ── Model + Data Paths ──
MODEL_PATH = "mistralai/Ministral-3-3B-Instruct-2512"
CKPT_OUTPUT_DIR = "./ministral_sft_output/checkpoints"

# ── Training Hyperparameters ──
NUM_EPOCHS = 3
EFFECTIVE_BATCH_SIZE = 32
LEARNING_RATE = 1e-5
MAX_SEQ_LEN = 4096
WARMUP_STEPS = 0

# ── Performance ──
MAX_TOKENS_PER_GPU = 8192

# ── Checkpointing ──
CHECKPOINT_AT_EPOCH = True
SAVE_SAMPLES = 0  # disable sample-based checkpoints; save at epoch end only

# ── Torchrun / Multi-GPU ──
NUM_GPUS = 8
NNODES = 1
NODE_RANK = 0
RDZV_ID = 300
RDZV_ENDPOINT = "localhost:42069"

available_gpus = torch.cuda.device_count()
if NUM_GPUS > available_gpus:
    raise RuntimeError(
        f"NUM_GPUS={NUM_GPUS}, but only {available_gpus} CUDA devices are visible. "
        "Lower NUM_GPUS before running training."
    )

print(f"Model:             {MODEL_PATH}")
print(f"Data:              {DATA_PATH} ({SUBSET_SIZE} samples)")
print(f"Output:            {CKPT_OUTPUT_DIR}")
print(f"Epochs:            {NUM_EPOCHS}")
print(f"Effective batch:   {EFFECTIVE_BATCH_SIZE}")
print(f"Learning rate:     {LEARNING_RATE}")
print(f"GPUs:              {NUM_GPUS}")
print(f"Max tokens/GPU:    {MAX_TOKENS_PER_GPU:,}")

## Training with SFT

This cell launches distributed training via torchrun. Training output is captured to keep the notebook clean — check the log file in `CKPT_OUTPUT_DIR` for full details.

In [ ]:
start_time = time.time()

captured_out = StringIO()
captured_err = StringIO()

try:
    with redirect_stdout(captured_out), redirect_stderr(captured_err):
        sft(
            model_path=MODEL_PATH,
            data_path=DATA_PATH,
            ckpt_output_dir=CKPT_OUTPUT_DIR,
            num_epochs=NUM_EPOCHS,
            effective_batch_size=EFFECTIVE_BATCH_SIZE,
            learning_rate=LEARNING_RATE,
            max_seq_len=MAX_SEQ_LEN,
            max_tokens_per_gpu=MAX_TOKENS_PER_GPU,
            data_output_dir=os.path.join(CKPT_OUTPUT_DIR, "_data"),
            warmup_steps=WARMUP_STEPS,
            save_samples=SAVE_SAMPLES,
            checkpoint_at_epoch=CHECKPOINT_AT_EPOCH,
            accelerate_full_state_at_epoch=False,
            nproc_per_node=NUM_GPUS,
            nnodes=NNODES,
            node_rank=NODE_RANK,
            rdzv_id=RDZV_ID,
            rdzv_endpoint=RDZV_ENDPOINT,
        )

    elapsed = time.time() - start_time
    ckpt = find_most_recent_checkpoint(CKPT_OUTPUT_DIR)
    if ckpt is None:
        raise FileNotFoundError(f"No checkpoint found under {CKPT_OUTPUT_DIR}")

    print(f"Training completed in {elapsed/60:.1f} minutes")
    print(f"Checkpoint saved to: {ckpt}")

except Exception as e:
    elapsed = time.time() - start_time
    print(f"Training failed after {elapsed/60:.1f} minutes")
    print(f"Error: {e}")
    print("\nStderr output:")
    print(captured_err.getvalue()[-2000:])
    raise

## Testing the Trained Model

Let's load the SFT checkpoint and test it on medical questions. The trained model should produce concise, factual answers matching the flashcard style — no markdown formatting, no verbose disclaimers.

In [ ]:
import torch
from transformers import Ministral3ForCausalLM, AutoTokenizer

ckpt_path = find_most_recent_checkpoint(CKPT_OUTPUT_DIR)
if ckpt_path is None:
    raise FileNotFoundError("Run the training cell successfully before loading a checkpoint.")
print(f"Loading checkpoint: {ckpt_path}")

tokenizer = AutoTokenizer.from_pretrained(ckpt_path)
model = Ministral3ForCausalLM.from_pretrained(
    ckpt_path,
    torch_dtype=torch.bfloat16,
    device_map="cuda:0",
)
model.eval()

test_questions = [
    "What are the classic signs and symptoms of diabetic ketoacidosis?",
    "What is the mechanism of action of ACE inhibitors in treating hypertension?",
    "What is the most common cause of iron deficiency anemia in adults?",
    "What are the differences between Type 1 and Type 2 diabetes mellitus?",
]

for i, question in enumerate(test_questions):
    messages = [{"role": "user", "content": question}]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=True,
            temperature=0.1,
            top_p=0.95,
        )

    response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)
    print(f"\nQ{i+1}: {question}")
    print(f"A{i+1}: {response.strip()}")
    print("-" * 70)

## Summary

In this notebook we:

1. **Loaded** the medical flashcards dataset and converted it to JSONL messages format
2. **Fine-tuned** Ministral 3 3B using SFT on 3,000 medical Q&A pairs
3. **Verified** the trained model produces concise, factual medical answers

**What to expect:** After SFT, the model shifts from verbose general-purpose responses to direct, textbook-style medical answers matching the flashcard training data.

**Next steps:**
- Try OSFT (`runnable_ministral_osft.ipynb`) for domain adaptation with knowledge retention
- Try LoRA (`runnable_ministral_lora.ipynb`) for parameter-efficient fine-tuning on a single GPU
- Scale up to the full 33,955-sample dataset for stronger domain adaptation
- Use `plot_loss()` from Training Hub to visualize the training curve